# EDA 2. Pret + cerere energie - piata din Spania

**Set de date:** `energy_dataset.csv` (29 coloane: generare, cerere, pret) + `weather_features.csv` (meteo pe 5 orase).

**Scop:** intelegerea relatiei dintre cerere, productie din diferite surse, conditii meteo si pretul de piata. Pretul determina decizia prescriptiva: incarcam bateria cand pretul e mic, vindem cand pretul e mare.

**Sursa originala:** ENTSOE Transparency Platform (operatorul european de retele de transport).

## Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data_processing.loader import load_pret_spania, merge_spania
from src.utils.plotting import (
    setup_style,
    plot_timeseries_resampled,
    plot_distribution,
    plot_seasonal_patterns,
    plot_correlation_heatmap,
    plot_missing_values,
)

setup_style()

## 1. Incarcarea datelor

Avem doua fisiere: unul cu datele energetice (orare, fara orase) si unul cu meteo (orar, pe cinci orase: Madrid, Barcelona, Valencia, Seville, Bilbao). Le incarcam separat si le analizam fiecare.

In [ ]:
energy, weather = load_pret_spania()

print(f'Energy: {energy.shape}, perioada {energy.index.min()} -> {energy.index.max()}')
print(f'Weather: {weather.shape}, orase: {weather["city_name"].unique().tolist()}')
energy.head(3)

**Interpretare:** datele acopera 4 ani (2015-2018), granularitate orara. Pentru meteo avem 5 orase x ~35k ore = ~178k randuri. Vom alege un singur oras (Madrid - capitala) pentru analiza univariata, iar la modelare putem agrega media pe toate orasele sau folosi orasele ca features separate.

## 2. Valori lipsa - dataset energetic

In [ ]:
ax = plot_missing_values(energy, title='Procent valori lipsa pe coloane (energy)')
plt.show()

**Interpretare:** doua coloane sunt aproape complet goale - `generation hydro pumped storage aggregated` si `forecast wind offshore eday ahead`. Le vom elimina la preprocesare. Restul coloanelor au sub 1% valori lipsa, care se pot completa prin interpolare.

In [ ]:
# Eliminam coloanele goale pentru analiza ulterioara
energy = energy.dropna(axis=1, how='all')
energy = energy.drop(columns=['forecast wind offshore eday ahead'], errors='ignore')
print(f'Coloane ramase: {energy.shape[1]}')

## 3. Pretul - variabila tinta

`price actual` este pretul realizat (EUR/MWh) si reprezinta tinta principala pentru recomandarile prescriptive.

In [ ]:
energy['price actual'].describe().round(2)

In [ ]:
ax = plot_timeseries_resampled(energy['price actual'], rule='D', title='Pret zilnic mediu (EUR/MWh)', ylabel='EUR/MWh')
plt.show()

In [ ]:
ax = plot_distribution(energy['price actual'], title='Distributie pret orar', xlabel='EUR/MWh')
plt.show()

**Interpretare:** pretul mediu este in jur de 57 EUR/MWh, cu o variatie larga (10-100+ EUR). Distributia este asimetrica spre dreapta - exista varfuri de pret rare dar foarte mari, care sunt exact momentele in care strategia prescriptiva (a vinde din baterie) este cea mai profitabila.

## 4. Cererea de energie (total load actual)

Comparabila cu PJME din USA, dar pe alta scara - Spania este mai mica.

In [ ]:
fig, axes = plot_seasonal_patterns(energy['total load actual'], title='Cerere Spania - tipare sezoniere (MW)')
plt.show()

**Interpretare:** spre deosebire de PJM (USA), in Spania varful de consum este la pranz si la prima ora de seara. Iarna are cerere mai mare datorita incalzirii, dar diferenta sezoniera nu este la fel de pronuntata ca in USA pentru ca clima este mai blanda.

## 5. Mixul de generare

Cat din energie vine din fiecare sursa? Aceasta este informatie cheie pentru a intelege presiunea pe pret.

In [ ]:
gen_cols = [c for c in energy.columns if c.startswith('generation') and 'storage' not in c]
mix = energy[gen_cols].mean().sort_values(ascending=True)
mix_pct = (mix / mix.sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(mix_pct.index.str.replace('generation ', ''), mix_pct.values, color='#2E5C8A')
ax.set_xlabel('Procent din mixul total (%)')
ax.set_title('Distributia surselor de energie in Spania (medie 2015-2018)')
for i, v in enumerate(mix_pct.values):
    if v > 0.5:
        ax.text(v + 0.3, i, f'{v}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

**Interpretare:** ponderile dominante sunt nuclear, gaze, eolian onshore si hidroelectric. Solar contribuie procente mici dar cu mare variabilitate (zi/noapte, sezon). Distributia este tipica pentru Spania - mix diversificat cu pondere semnificativa de regenerabile.

## 6. Corelatii intre variabilele cheie

Selectam un subset reprezentativ de coloane pentru a vedea cum se influenteaza reciproc.

In [ ]:
key_cols = [
    'price actual', 'total load actual',
    'generation fossil gas', 'generation nuclear',
    'generation wind onshore', 'generation solar',
    'generation hydro water reservoir',
]
ax = plot_correlation_heatmap(energy[key_cols], title='Corelatii pret - cerere - generare')
plt.show()

**Interpretare:**
- Pretul are corelatie pozitiva moderata cu cererea totala (cand creste cererea, creste si pretul - logica de piata).
- Pretul este corelat pozitiv cu generarea fosila pe gaz - centralele pe gaz sunt setterii de pret in orele de varf.
- Generarea eoliana si solara au corelatie negativa cu pretul - cand bate vantul / e soare, energia e mai ieftina.
- Acesta este insight-ul cheie: regenerabilele apasa pretul in jos, fosilele il urca.

## 7. Combinare cu meteo (Madrid)

Folosim functia `merge_spania` care selecteaza un singur oras si face join cu datele energetice.

In [ ]:
merged = merge_spania(city='Madrid')
# Convertim temperatura din Kelvin in Celsius
merged['temp_C'] = merged['temp'] - 273.15

weather_cols = ['price actual', 'total load actual', 'temp_C', 'humidity', 'wind_speed', 'clouds_all']
ax = plot_correlation_heatmap(merged[weather_cols], title='Corelatii pret - cerere - meteo (Madrid)')
plt.show()

**Interpretare:** temperatura are corelatie cu cererea (mai cald + climatizare sau mai frig + incalzire), iar viteza vantului influenteaza pretul indirect prin generarea eoliana. Aceste features meteo sunt foarte utile pentru forecasting.

## 8. Concluzii si pasi urmatori

**Ce am invatat:**
1. Datele acopera 2015-2018, orar, cu 5 orase de meteo.
2. Pretul prezinta sezonalitate si volatilitate, varfuri pretioase pentru strategia prescriptiva.
3. Mixul energetic este diversificat; eolianul si solarul scad pretul, fosilele il urca.
4. Meteo (temperatura, vant) este corelat cu cererea si pretul.

**Pasi urmatori:**
- Eliminare coloane goale; interpolare pe coloanele cu < 1% NaN.
- Feature engineering meteo (agregare pe orase sau features separate per oras).
- Construire model predictiv pentru `price actual` si pentru `total load actual`.
- Definire problema de optimizare: dispatch baterie cu obiectiv max profit pe 24h.

Acest set este in mod special pretios pentru componenta de optimizare neliniara - avem semnale clare de pret pentru a defini problema prescriptiva.